<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; EDM&plus; Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">EDM&plus; NB01 &mdash; Foundation Setup</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Stands up the EDM&plus; data model: custom data types, property definitions across five domains, and coalescing derived properties with override support.</div>
</div>

<sub>EDM&plus; pack sequence: <b>NB01</b> &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07</sub>

# NB01: Foundation Setup
    
**What this does:** Creates the entire data model foundation for EDM+. Without this, nothing else works.

**Business context:** Every EDM implementation starts by defining the data model. This means creating custom data types (constrained value lists that enforce data quality), property definitions (custom attributes on instruments, portfolios, transactions, and legal entities), and derived properties (calculated fields that implement golden source override logic).

**What you will build:**
- 6 custom data types (e.g. CreditRatingScale with AAA through D)
- ~60 property definitions across 5 domains
- Override properties in separate scopes for manual corrections
- 5 derived properties with coalesce/override formulas

**Verify after running:** Go to Settings → Properties in the LUSID web app. Filter by scope `edm-training` to see your properties.

In [ ]:
# Run this cell first — installs required packages (takes ~30 seconds, only needed once per session)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'lusid-sdk', 'lusid-workflow-sdk', 'finbourne-sdk-utilities', 'lusid-drive-sdk', 'lumipy'])
print("✅ Packages installed")

In [ ]:
# ============================================================================
# EDIT THESE TWO LINES with your domain and PAT token
# ============================================================================
# To get a PAT: LUSID web app → profile icon (top right) → Personal Access Tokens → Create
# Replace <YOUR_DOMAIN> with your domain name (e.g. "mignot" or "acme-corp")
# Replace <YOUR_PAT> with your actual token string

api_url = "https://<YOUR_DOMAIN>.lusid.com/api"
personal_access_token = "<YOUR_PAT>"

# ============================================================================
# Imports — do not edit below this line
# ============================================================================
import os, json
from datetime import datetime, timedelta, date, timezone
import datetime as dt
import pandas as pd
import pytz

import lusid as lu
import lusid.models as lm

try:
    import lusid_workflow as lwf
    import lusid_workflow.models as wf_models
except ImportError:
    lwf = None
    wf_models = None
    print("⚠️ lusid_workflow not available")

import lumipy as lp

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.4f}".format

# ============================================================================
# Authentication
# ============================================================================
if "<YOUR_DOMAIN>" in api_url or "<YOUR_PAT>" in personal_access_token:
    raise ValueError(
        "\n\n⛔ You need to edit the two lines at the top of this cell:\n"
        "   1. Replace <YOUR_DOMAIN> with your LUSID domain name\n"
        "   2. Replace <YOUR_PAT> with your Personal Access Token\n\n"
        "   To get a PAT: LUSID web app → profile icon → Personal Access Tokens → Create")

def get_factory(app='lusid'):
    module = __import__(app)
    # Each service has a different URL path
    url_map = {'lusid': '/api', 'lusid_workflow': '/workflow', 'lusid_drive': '/drive'}
    service_url = api_url.replace('/api', url_map.get(app, '/api'))
    config_loaders = [module.extensions.ArgsConfigurationLoader(
        api_url=service_url, access_token=personal_access_token)]
    return module.extensions.SyncApiClientFactory(config_loaders=config_loaders)

def get_lumi():
    luminesce_url = api_url.replace('/api', '/honeycomb')
    return lp.get_client(access_token=personal_access_token, api_url=luminesce_url)

# Initialise
lusid_factory = get_factory('lusid')

# Verify connection
try:
    api_instance = lusid_factory.build(lu.ApplicationMetadataApi)
    api_response = api_instance.get_lusid_versions()
    domain = api_response.links[0].href
    env_url = domain[0:domain.find('/app/')]
    print(f"✅ Connected to {env_url}")
    print(f"   API Version: {api_response.build_version}")
except Exception as e:
    print(f"⚠️ Connected but could not verify: {e}")

lumi = get_lumi()
print("✅ Luminesce ready")

---
## Build APIs

In [ ]:
property_definitions_api = lusid_factory.build(lu.PropertyDefinitionsApi)
data_types_api = lusid_factory.build(lu.DataTypesApi)

SCOPE = 'edm-training2'
SOURCE_SCOPE_OVERRIDE = 'Override'
SOURCE_SCOPE_MANUAL = 'Manual'
CALC_SCOPE_MASTER = 'Master'
PORTFOLIO_SCOPE = f'{SCOPE}-accounts'

print(f"Primary scope: {SCOPE}")
print(f"Portfolio scope: {PORTFOLIO_SCOPE}")

---
## Helper Functions

In [ ]:
def create_data_type(scope, code_val, display_name, description, acceptable_values):
    try:
        data_types_api.create_data_type(
            create_data_type_request=lm.CreateDataTypeRequest(
                scope=scope, code=code_val, type_value_range='Closed',
                display_name=display_name, description=description,
                value_type='String', acceptable_values=acceptable_values))
        print(f'  ✅ Created: {scope}/{code_val} ({len(acceptable_values)} values)')
    except lu.ApiException as e:
        if 'AlreadyExists' in str(e.body):
            print(f'  ℹ️  Exists: {scope}/{code_val}')
        else:
            error = json.loads(e.body)
            print(f'  ❌ Error: {error.get("title", e.reason)}')

def create_property(domain, scope, code_val, display_name, data_type='string',
                    description='', constraint_style='Property', lifetime='Perpetual'):
    try:
        property_definitions_api.create_property_definition(
            create_property_definition_request=lm.CreatePropertyDefinitionRequest(
                domain=domain, scope=scope, code=code_val,
                display_name=display_name,
                data_type_id=lm.ResourceId(scope='system', code=data_type),
                description=description or display_name,
                constraint_style=constraint_style, life_time=lifetime))
        print(f'  ✅ Created: {domain}/{scope}/{code_val}')
    except lu.ApiException as e:
        if 'AlreadyExists' in str(e.body):
            print(f'  ℹ️  Exists: {domain}/{scope}/{code_val}')
        else:
            error = json.loads(e.body)
            print(f'  ❌ Error: {error.get("title", e.reason)}')

def create_derived_property(domain, scope, code_val, display_name, formula, description=''):
    try:
        property_definitions_api.create_derived_property_definition(
            create_derived_property_definition_request=lm.CreateDerivedPropertyDefinitionRequest(
                domain=domain, scope=scope, code=code_val,
                display_name=display_name,
                data_type_id=lm.ResourceId(scope='system', code='string'),
                description=description or display_name,
                derivation_formula=formula,
                is_filterable=True))
        print(f'  ✅ Created derived: {domain}/{scope}/{code_val}')
    except lu.ApiException as e:
        if 'AlreadyExists' in str(e.body):
            print(f'  ℹ️  Exists derived: {domain}/{scope}/{code_val}')
        else:
            error = json.loads(e.body)
            print(f'  ❌ Error: {error.get("title", e.reason)}')

---
## Create Custom Data Types

These constrained value lists enforce data quality. For example, if someone tries to set a credit rating to an invalid value, LUSID rejects it.

In [ ]:
print("--- Creating Custom Data Types ---")
create_data_type(SCOPE, 'DataQualityStatus', 'Data Quality Status', 'Quality review state',
    ['Completed','New','PendingUpdate','UnderReview','Rejected','Bronze','Silver','Gold'])
create_data_type(SCOPE, 'CreditRatingScale', 'Credit Rating Scale', 'Credit ratings',
    ['AAA','AA+','AA','AA-','A+','A','A-','BBB+','BBB','BBB-','BB+','BB','BB-',
     'B+','B','B-','CCC+','CCC','CCC-','CC','C','D','NR','WD'])
create_data_type(SCOPE, 'AssetClassification', 'Asset Classification', 'Asset class taxonomy',
    ['Common Stock','Government Bond','Corporate Bond','ETF','Equity Option','FX Forward',
     'Future','Private Equity','Infrastructure','Real Estate','Cash','Repo','Fund',
     'Convertible','ABS','Interest Rate Swap','Credit Default Swap'])
create_data_type(SCOPE, 'ESGRatingScale', 'ESG Rating Scale', 'ESG 0 to 100',
    [str(i) for i in range(0, 101)])
create_data_type(SCOPE, 'RiskProfile', 'Risk Profile', 'Risk level 1 to 5',
    ['1','2','3','4','5'])
create_data_type(SCOPE, 'WatchListStatus', 'Watch List Status', 'On watch list',
    ['TRUE','FALSE'])

---
## Create Property Definitions

Properties extend LUSID entities with custom attributes. We create them across Instrument, Portfolio, Transaction, and LegalEntity domains.

In [ ]:
print("--- Instrument Properties ---")
for code_val, name, dt_val, desc in [
    ('Sector','Sector','string','Industry sector'),
    ('Exchange','Exchange','string','Primary exchange'),
    ('ListingCountry','Listing Country','string','Country of listing'),
    ('DividendYield','Dividend Yield','number','Current dividend yield'),
    ('MarketCapitalization','Market Capitalisation','number','Market cap'),
    ('SharesOutstanding','Shares Outstanding','number','Total shares'),
    ('VotingRights','Voting Rights','string','Voting rights type'),
    ('BondType','Bond Type','string','Bond type'),
    ('CouponRate','Coupon Rate','number','Coupon rate'),
    ('CouponFrequency','Coupon Frequency','string','Payment frequency'),
    ('CreditRating','Credit Rating','string','Primary rating'),
    ('DayCountConvention','Day Count Convention','string','Day count'),
    ('FaceValue','Face Value','number','Par value'),
    ('YieldToMaturity','Yield to Maturity','number','YTM'),
    ('CallableFlag','Callable Flag','string','Callable'),
    ('PuttableFlag','Puttable Flag','string','Puttable'),
    ('ConvertibleFlag','Convertible Flag','string','Convertible'),
    ('AccruedInterest','Accrued Interest','number','Accrued interest'),
    ('Subordination','Subordination','string','Debt subordination'),
    ('Covenants','Covenants','string','Covenant type'),
    ('DefaultProbability','Default Probability','number','PD'),
    ('RecoveryRate','Recovery Rate','number','Recovery rate'),
    ('NaceLevel1','NACE Level 1','string','NACE classification'),
    ('CountryIso','Country ISO','string','ISO country code'),
    ('DomicileCountry','Domicile Country','string','Domicile'),
    ('IssuerName','Issuer Name','string','Issuing entity'),
    ('AUM','AUM','number','Assets under management'),
    ('ExpenseRatio','Expense Ratio','number','Fund expense ratio'),
    ('UnderlyingIndex','Underlying Index','string','Tracked index'),
    ('Replication','Replication Method','string','Replication method'),
]:
    create_property('Instrument', SCOPE, code_val, name, dt_val, desc)

In [ ]:
print("--- ESG / Ratings / Governance Properties ---")
for code_val, name, dt_val, desc in [
    ('EsgEnvironmentRating','ESG Environment Rating','number','E score'),
    ('EsgSocialRating','ESG Social Rating','number','S score'),
    ('EsgGovernanceRating','ESG Governance Rating','number','G score'),
    ('DataQualityStatus','Data Quality Status','string','DQ state'),
    ('DataOwner','Data Owner','string','Data owner'),
    ('DataGovernanceCode','Data Governance Code','string','Governance code'),
    ('WatchList','Watch List','string','On watch list'),
    ('RatingS','S&P Rating','string','S&P rating'),
    ('RatingM','Moodys Rating','string','Moodys rating'),
    ('RatingF','Fitch Rating','string','Fitch rating'),
    ('RiskProfile','Risk Profile','string','Risk level'),
]:
    create_property('Instrument', SCOPE, code_val, name, dt_val, desc)

In [ ]:
print("--- Portfolio Properties ---")
for code_val, name, dt_val, desc in [
    ('Strategy','Strategy','string','Investment strategy'),
    ('RiskProfile','Risk Profile','string','Portfolio risk level'),
    ('CustodianAccount','Custodian Account','string','Custodian account'),
    ('PortfolioManager','Portfolio Manager','string','PM'),
    ('Benchmark','Benchmark','string','Benchmark index'),
    ('InceptionDate','Inception Date','string','Inception date'),
    ('AssetClass','Asset Class','string','Primary asset class'),
]:
    create_property('Portfolio', PORTFOLIO_SCOPE, code_val, name, dt_val, desc)

print("\n--- Transaction Properties ---")
for code_val, name, dt_val, desc in [
    ('Strategy','Strategy','string','Trade strategy'),
    ('CustodianAccount','Custodian Account','string','Settlement custodian'),
    ('BalanceSheetAllocation','Balance Sheet Allocation','string','BS line item'),
    ('ShareClassId','Share Class ID','string','Share class'),
    ('Broker','Broker','string','Executing broker'),
]:
    create_property('Transaction', SCOPE, code_val, name, dt_val, desc)

print("\n--- Legal Entity Properties ---")
for code_val, name, dt_val, desc in [
    ('LEI','LEI','string','Legal Entity Identifier'),
    ('Industry','Industry','string','Industry'),
    ('RegulatoryStatus','Regulatory Status','string','Regulatory status'),
    ('TaxResidency','Tax Residency','string','Tax jurisdiction'),
    ('RegistrationNumber','Registration Number','string','Company registration'),
    ('Domicile','Domicile','string','Country of domicile'),
    ('IssuerCountry','Issuer Country','string','Issuer country'),
    ('Address','Address','string','Registered address'),
    ('SolvencyIIRating','Solvency II Rating','string','Solvency II rating'),
    ('CounterpartyType','Counterparty Type','string','Counterparty type'),
]:
    create_property('LegalEntity', SCOPE, code_val, name, dt_val, desc)

---
## Override Properties

These are created in separate scopes (`Override` and `Manual`) so business users can correct individual values without modifying the automated feed. They must be created before derived properties because the formulas reference them.

In [ ]:
print("--- Override Properties ---")
for code_val, name, dt_val, desc in [
    ('AssetClassifier_Override','Asset Classifier Override','string','Manual override'),
    ('IssuerName_Override','Issuer Name Override','string','Manual override'),
    ('CreditRating_Override','Credit Rating Override','string','Manual override'),
    ('DataQualityStatus','Data Quality Status Override','string','Manual override'),
]:
    create_property('Instrument', SOURCE_SCOPE_OVERRIDE, code_val, name, dt_val, desc)
    create_property('Instrument', SOURCE_SCOPE_MANUAL, code_val, name, dt_val, desc)

---
## Derived Properties

These are the golden source formulas. Each uses `coalesce()` to check an override scope first, then fall back to the automated source value. LUSID calculates these on the fly at query time.

For example, `MasterCreditRating` checks S&P first, then Moody's, then Fitch, then the generic CreditRating field.

In [ ]:
print("--- Derived Properties ---")
create_derived_property('Instrument', CALC_SCOPE_MASTER, 'AssetClassifier',
    'Master Asset Classifier',
    f"coalesce(Properties[Instrument/{SOURCE_SCOPE_OVERRIDE}/AssetClassifier_Override], Properties[Instrument/{SCOPE}/Sector])",
    'Golden source with override')

create_derived_property('Instrument', CALC_SCOPE_MASTER, 'MasterCreditRating',
    'Master Credit Rating',
    f"coalesce(Properties[Instrument/{SCOPE}/RatingS], Properties[Instrument/{SCOPE}/RatingM], Properties[Instrument/{SCOPE}/RatingF], Properties[Instrument/{SCOPE}/CreditRating])",
    'Preferred rating across agencies')

create_derived_property('Instrument', CALC_SCOPE_MASTER, 'DataQualityFlag',
    'Data Quality Flag',
    f"coalesce(Properties[Instrument/{SOURCE_SCOPE_MANUAL}/DataQualityStatus], Properties[Instrument/{SCOPE}/DataQualityStatus], 'New')",
    'DQ status with fallback')

create_derived_property('Instrument', CALC_SCOPE_MASTER, 'MasterCountry',
    'Master Country',
    f"coalesce(Properties[Instrument/{SCOPE}/CountryIso], Properties[Instrument/{SCOPE}/DomicileCountry], Properties[Instrument/{SCOPE}/ListingCountry])",
    'Country resolution')

create_derived_property('Instrument', CALC_SCOPE_MASTER, 'MasterIssuer',
    'Master Issuer',
    f"coalesce(Properties[Instrument/{SOURCE_SCOPE_OVERRIDE}/IssuerName_Override], Properties[Instrument/{SCOPE}/IssuerName])",
    'Issuer with override')

print("\n✅ NB01 Complete")